### Building BPE from Scratch 

Before a Large Language Model (LLM) calculates a single attention weight or generates a line of code, it has to do something more fundamental: read :)

But language models don't understand words, punctuation, or poetic nuance the way we do. At their core, they speak numbers.

Enter the tokenizer that breaks the raw text into smaller units called tokens and maps them to numerical IDs that the model can process.

If you look under the hood of today's state-of-the-art models from OpenAI's GPT-4 to Meta's Llama 3, you'll find that almost all of them rely on a specific algorithm called Byte Pair Encoding (BPE). But BPE wasn't designed for artificial intelligence, transformers, or neural networks.

To understand why every major AI lab uses BPE today, we have to look back thirty years.

A Brief History : In 1994, developer Philip Gage published a paper in the C/C++ Users Journal titled "A New Algorithm for Data Compression". His goal had nothing to do with natural language processing; he was looking for a simple, fast way to shrink file sizes. His trick was simple: scan a body of text, find the most frequently occurring pair of adjacent bytes, replace that pair with an unused single byte, and repeat.

For two decades, BPE remained a clever software utility for file compression.

Fast forward to 2015. Researchers working on Neural Machine Translation faced a massive bottleneck: how do you handle rare words, typos, and infinite vocabulary?

Rico Sennrich and his team realized that Philip Gage’s 20-year-old compression algorithm was actually the missing link for natural language. By adapting BPE to text processing, they proved that language models didn't need to choose between whole words or individual characters—they could break language down into reusable subwords. OpenAI later refined this into Byte-Level BPE for GPT-2, solidifying it as the industry standard for LLM tokenization.

Part 1: UTF-8 Byte Conversions & Unicode Code Point Mapping

Tokenization is the critical bridge between raw human text and numerical tensor inputs in Large Language Models (LLMs). Before a Transformer computes a single attention weight, text must be discretized into a sequence of integer token IDs.

Historically, Natural Language Processing (NLP) pipelines faced a fundamental dilemma:
- Character-Level Tokenization: Keeps vocabulary size very small ($V \approx 100\text{--}1,000$), but causes input sequence length ($N$) to explode. Because Transformer self-attention scales quadratically ($O(N^2)$) in memory and time, context window costs become prohibitively expensive.

- Word-Level Tokenization: Yields short sequence lengths ($N$), but causes vocabulary size ($V$) to balloon into hundreds of thousands of entries. Worse, it routinely encounters Out-Of-Vocabulary (OOV) words, forcing the tokenizer to output lossy [UNK] (unknown) tokens whenever it encounters typos, rare slang, or domain-specific jargon.

Byte-Level Byte-Pair Encoding (BPE)—the core algorithm powering GPT-2, GPT-4, LLaMA, and Mistral—resolves this trade-off. By starting tokenization at the raw byte level rather than the character level, we establish an initial base vocabulary of exactly $V_0 = 256$ tokens. From there, BPE iteratively merges the most frequent adjacent byte pairs into richer subword units.

To understand why byte-level BPE is so powerful, we must distinguish between Unicode Code Points and UTF-8 Byte Encodings.
- Unicode Code Point: An abstract numerical index assigned to every character in the  [Unicode standard ](https://en.wikipedia.org/wiki/Unicode) (which contains over 149,000 characters). For example, the character 'A' is U+0041, the Devanagari character 'अ' is U+0905, and the emoji '🤖' is U+1F916.

- UTF-8: A variable-length byte encoding system that converts Unicode code points into sequences of 8-bit bytes (integers ranging from $0$ to $255$).

| Text Input | Character Count | Code Point (Hex) | UTF-8 Byte Sequence (Hex) | UTF-8 Byte Integers (0–255) | Byte Length |
|---|---:|---|---|---|---:|
| `"A"` | 1 | `U+0041` | `0x41` | `[65]` | **1 byte** |
| `"अ"` (Devanagari) | 1 | `U+0905` | `0xE0 0xA4 0x85` | `[224, 164, 133]` | **3 bytes** |
| `"🤖"` (Emoji) | 1 | `U+1F916` | `0xF0 0x9F 0xA4 0x96` | `[240, 159, 164, 150]` | **4 bytes** |

In [18]:
def analyze_utf8_encoding(text: str) -> list[int]:
    """Analyzes and prints UTF-8 byte encoding breakdown for a given string."""
    raw_bytes = text.encode("utf-8")
    tokens = list(raw_bytes)

    print(f"Original Text:            {text!r}")
    print(f"Byte Integers (0-255):    {tokens}")
    print("-" * 60)

    return tokens

if __name__ == "__main__":
    # Experimenting with ASCII, multi-byte Devanagari, and Emojis
    tokens1 = analyze_utf8_encoding("Hello World!")  # ASCII
    tokens2 = analyze_utf8_encoding("नमस्ते")  # Devanagari "Namaste"
    tokens3 = analyze_utf8_encoding("🙏")     # Emoji + ASCII

Original Text:            'Hello World!'
Byte Integers (0-255):    [72, 101, 108, 108, 111, 32, 87, 111, 114, 108, 100, 33]
------------------------------------------------------------
Original Text:            'नमस्ते'
Byte Integers (0-255):    [224, 164, 168, 224, 164, 174, 224, 164, 184, 224, 165, 141, 224, 164, 164, 224, 165, 135]
------------------------------------------------------------
Original Text:            '🙏'
Byte Integers (0-255):    [240, 159, 153, 143]
------------------------------------------------------------


In [17]:
def decode_utf8_bytes(byte_integers: list[int]) -> str:
    """Decodes a list of UTF-8 byte integers (0-255) back into a Unicode string."""
    byte_array = bytes(byte_integers)
    return byte_array.decode("utf-8")

if __name__ == "__main__":
    # Test using tokens from previous encodings
    print(f"Decoded Text 1: {decode_utf8_bytes(tokens1)!r}")
    print(f"Decoded Text 2: {decode_utf8_bytes(tokens2)!r}")
    print(f"Decoded Text 3: {decode_utf8_bytes(tokens3)!r}")  

Decoded Text 1: 'Hello World!'
Decoded Text 2: 'नमस्ते'
Decoded Text 3: '🤖 AI'


---

In [ ]:
import json
import os
import regex as re

# Standard GPT-2 split pattern for pre-tokenization
GPT2_SPLIT_PATTERN = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""


def get_stats(ids, counts=None):
    """
    Counts occurrences of adjacent token ID pairs.
    """
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids, pair, idx):
    """
    Replaces all non-overlapping occurrences of `pair` in `ids` with `idx`.
    """
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


class ByteLevelBPETokenizer:
    def __init__(self, pattern=None):
        self.pattern = GPT2_SPLIT_PATTERN if pattern is None else pattern
        self.compiled_pattern = re.compile(self.pattern)
        self.merges = {}  # tuple (p0, p1) -> merged_token_id
        self.vocab = {i: bytes([i]) for i in range(256)}  # int -> bytes
        self.special_tokens = {}  # str -> token_id
        self.inverse_special_tokens = {}  # token_id -> str

    def add_special_tokens(self, special_tokens):
        """
        Registers special tokens (e.g., {"<|endoftext|>": 100000}).
        """
        for token, idx in special_tokens.items():
            self.special_tokens[token] = idx
            self.inverse_special_tokens[idx] = token
            self.vocab[idx] = token.encode("utf-8")

    def train(self, text, vocab_size, verbose=False):
        """
        Trains the BPE tokenizer on raw text up to the specified vocabulary size.
        """
        assert vocab_size >= 256, "Vocabulary size must be at least 256."
        num_merges = vocab_size - 256

        # Step 1: Split text into chunks using pre-tokenization regex
        text_chunks = self.compiled_pattern.findall(text)

        # Step 2: Convert chunks into lists of raw UTF-8 byte integer IDs
        ids_chunks = [list(chunk.encode("utf-8")) for chunk in text_chunks]

        # Step 3: Iteratively find the most frequent pair and merge
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(256)}

        for i in range(num_merges):
            stats = {}
            for chunk in ids_chunks:
                get_stats(chunk, stats)

            if not stats:
                break

            # Find the most frequent adjacent pair
            best_pair = max(stats, key=stats.get)
            if stats[best_pair] < 1:
                break

            new_id = 256 + i
            ids_chunks = [merge(chunk, best_pair, new_id) for chunk in ids_chunks]

            # Save learned merge rank and vocabulary token
            self.merges[best_pair] = new_id
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]

            if verbose:
                print(f"Merge {i+1}/{num_merges}: {best_pair} -> {new_id} ({self.vocab[new_id]})")

        # Re-register special tokens if present
        for token, idx in self.special_tokens.items():
            self.vocab[idx] = token.encode("utf-8")

    def _encode_chunk(self, chunk_bytes):
        """
        Encodes a single pre-tokenized byte chunk applying learned merges in chronological rank order.
        """
        ids = list(chunk_bytes)
        while len(ids) >= 2:
            stats = get_stats(ids)
            # Find the pair with the lowest rank (learned earliest during training)
            pair = min(stats.keys(), key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            idx = self.merges[pair]
            ids = merge(ids, pair, idx)
        return ids

    def encode(self, text, allowed_special="none"):
        """
        Encodes text into integer token IDs.
        `allowed_special` can be "all", "none", or a set of explicit special token strings.
        """
        # Determine active special tokens for this encode call
        if allowed_special == "all":
            active_specials = self.special_tokens
        elif allowed_special == "none":
            active_specials = {}
        elif isinstance(allowed_special, set):
            active_specials = {k: v for k, v in self.special_tokens.items() if k in allowed_special}
        else:
            raise ValueError(f"Invalid allowed_special argument: {allowed_special}")

        if active_specials:
            # Escape and compile special token search pattern
            special_pattern = "(" + "|".join(re.escape(k) for k in active_specials.keys()) + ")"
            chunks = re.split(special_pattern, text)
        else:
            chunks = [text]

        ids = []
        for chunk in chunks:
            if chunk in active_specials:
                ids.append(active_specials[chunk])
            else:
                # Pre-tokenize standard text chunk using regex split rules
                sub_chunks = self.compiled_pattern.findall(chunk)
                for sub_chunk in sub_chunks:
                    chunk_bytes = sub_chunk.encode("utf-8")
                    ids.extend(self._encode_chunk(chunk_bytes))
        return ids

    def decode(self, ids):
        """
        Decodes a sequence of integer token IDs back to a text string.
        """
        part_bytes = []
        for idx in ids:
            if idx in self.vocab:
                part_bytes.append(self.vocab[idx])
            elif idx in self.inverse_special_tokens:
                part_bytes.append(self.inverse_special_tokens[idx].encode("utf-8"))
            else:
                raise ValueError(f"Invalid token ID: {idx}")

        return b"".join(part_bytes).decode("utf-8", errors="replace")

    def save(self, file_prefix):
        """
        Saves the tokenizer metadata, special tokens, and merge rules to disk.
        """
        model_file = file_prefix + ".model"
        with open(model_file, "w", encoding="utf-8") as f:
            f.write("byte-level-bpe v1\n")
            f.write(f"{self.pattern}\n")
            f.write(f"{len(self.special_tokens)}\n")
            for k, v in self.special_tokens.items():
                f.write(f"{k} {v}\n")
            for (p0, p1), idx in self.merges.items():
                f.write(f"{p0} {p1} {idx}\n")

    def load(self, model_file):
        """
        Loads trained merge rules and tokenizer configuration from disk.
        """
        with open(model_file, "r", encoding="utf-8") as f:
            version = f.readline().strip()
            self.pattern = f.readline().strip()
            self.compiled_pattern = re.compile(self.pattern)

            num_special = int(f.readline().strip())
            self.special_tokens = {}
            self.inverse_special_tokens = {}
            for _ in range(num_special):
                line = f.readline().strip().split()
                k, v = line[0], int(line[1])
                self.special_tokens[k] = v
                self.inverse_special_tokens[v] = k

            self.merges = {}
            self.vocab = {i: bytes([i]) for i in range(256)}
            for k, v in self.special_tokens.items():
                self.vocab[v] = k.encode("utf-8")

            for line in f:
                if line.strip():
                    p0, p1, idx = map(int, line.strip().split())
                    self.merges[(p0, p1)] = idx
                    self.vocab[idx] = self.vocab[p0] + self.vocab[p1]

if __name__ == "__main__":
    training_corpus = """
    Hello world! Building a Byte-Pair Encoding (BPE) tokenizer from scratch is fun.
    Unicode support check: नमस्ते दुनिया! Café, naïve, 123456789.
    Special token test: <|endoftext|>
    """

    # 1. Instantiate Tokenizer
    tokenizer = ByteLevelBPETokenizer()

    # 2. Add Special Tokens
    tokenizer.add_special_tokens({"<|endoftext|>": 1000})

    # 3. Train on Corpus
    target_vocab_size = 300  # 256 base byte tokens + 44 merges
    print(f"Training tokenizer to target vocab size: {target_vocab_size}...")
    tokenizer.train(training_corpus, vocab_size=target_vocab_size, verbose=False)

    # 4. Test Sample Texts
    test_text = "Hello world! <|endoftext|> नमस्ते world!"
    
    # Encode
    encoded_ids = tokenizer.encode(test_text, allowed_special="all")
    print("\nEncoded Token IDs:")
    print(encoded_ids)

    # Decode
    decoded_text = tokenizer.decode(encoded_ids)
    print("\nDecoded Text:")
    print(decoded_text)

    # Round-trip Assertion Test
    assert test_text == decoded_text, "Round-trip assertion failed!"
    print("\nRound-trip test passed successfully!")

    # 5. Serialization Test
    tokenizer.save("bpe_test")
    
    loaded_tokenizer = ByteLevelBPETokenizer()
    loaded_tokenizer.load("bpe_test.model")
    
    assert loaded_tokenizer.encode(test_text, allowed_special="all") == encoded_ids
    assert loaded_tokenizer.decode(encoded_ids) == test_text
    print("Serialization save/load verification passed!")

Training tokenizer to target vocab size: 300...

Encoded Token IDs:
[72, 101, 108, 108, 111, 286, 33, 32, 1000, 278, 168, 257, 174, 257, 184, 262, 141, 257, 164, 262, 135, 286, 33]

Decoded Text:
Hello world! <|endoftext|> नमस्ते world!

Round-trip test passed successfully!
Serialization save/load verification passed!


---

To upgrade your BPE tokenizer from GPT-2 style to GPT-4 (cl100k_base) or GPT-4o (o200k_base) standards, four major architecture and algorithmic improvements are required:

1. Updated GPT-4 Regex Pre-tokenization Pattern:

    - Case-Insensitive Contractions: Captures 'S, 's, 'RE, 're uniformly ((?i:...)).

    - Digit Chunking (\p{N}{1,3}): Restricts number merges to a maximum of 3 digits at a time. This stops long numerical sequences (e.g., 123456789) from merging into arbitrary single tokens, significantly boosting code and math capabilities.

    - Whitespace & Newline Isolation: Keeps trailing newlines and indentations separate to maintain syntax awareness for Python/YAML.

2. Expanded Vocabulary Scale: GPT-4 increases vocabulary size to ~100k tokens (cl100k_base), while GPT-4o expands to ~200k tokens (o200k_base). A larger vocabulary improves compression ratio by up to 2x for non-English languages and code.

3. Fill-In-the-Middle (FIM) & System Control Tokens: Full support for code completion control tokens (<|fim_prefix|>, <|fim_middle|>, <|fim_suffix|>, <|endofprompt|>) alongside standard sequence markers.

4. Rank-Based Encoding Optimizations: Accelerated chunk encoding using $O(1)$ priority lookups on trained merge ranks rather than re-computing pair frequency histograms.

In [29]:
import json
import os
import regex as re

# Official GPT-4 Pre-tokenization Regex Pattern (tiktoken cl100k_base)
GPT4_SPLIT_PATTERN = r"""(?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+"""

# Standard GPT-4 Special / Control Tokens
GPT4_SPECIAL_TOKENS = {
    "<|endoftext|>": 100000,
    "<|fim_prefix|>": 100001,
    "<|fim_middle|>": 100002,
    "<|fim_suffix|>": 100003,
    "<|endofprompt|>": 100004,
}


def get_stats(ids, counts=None):
    """
    Counts occurrences of adjacent token ID pairs.
    """
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids, pair, idx):
    """
    Replaces all non-overlapping occurrences of `pair` in `ids` with `idx`.
    """
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


class GPT4BPETokenizer:
    def __init__(self, pattern=None):
        self.pattern = GPT4_SPLIT_PATTERN if pattern is None else pattern
        self.compiled_pattern = re.compile(self.pattern)
        self.merges = {}  # tuple (p0, p1) -> token_id (lower token_id = earlier merge rank)
        self.vocab = {i: bytes([i]) for i in range(256)}  # int -> bytes
        self.special_tokens = {}
        self.inverse_special_tokens = {}

        # Register default GPT-4 special tokens
        self.register_special_tokens(GPT4_SPECIAL_TOKENS)

    def register_special_tokens(self, special_tokens_dict):
        """
        Registers control and special tokens with custom fixed token IDs.
        """
        for token, idx in special_tokens_dict.items():
            self.special_tokens[token] = idx
            self.inverse_special_tokens[idx] = token
            self.vocab[idx] = token.encode("utf-8")

    def train(self, text, vocab_size=100000, verbose=False):
        """
        Trains the BPE tokenizer on raw text up to the specified target vocabulary size.
        """
        assert vocab_size >= 256, "Vocabulary size must be at least 256."
        num_merges = vocab_size - 256

        # Step 1: Pre-tokenize text into structural chunks using GPT-4 regex rules
        text_chunks = self.compiled_pattern.findall(text)

        # Step 2: Convert chunks into raw UTF-8 byte integer lists
        ids_chunks = [list(chunk.encode("utf-8")) for chunk in text_chunks]

        # Step 3: Train BPE merges iteratively
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(256)}

        for i in range(num_merges):
            stats = {}
            for chunk in ids_chunks:
                get_stats(chunk, stats)

            if not stats:
                break

            # Select pair with highest global frequency
            best_pair = max(stats, key=stats.get)
            if stats[best_pair] < 1:
                break

            new_id = 256 + i
            ids_chunks = [merge(chunk, best_pair, new_id) for chunk in ids_chunks]

            # Save learned rank and vocabulary byte sequence
            self.merges[best_pair] = new_id
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]

            if verbose and (i + 1) % 1000 == 0:
                print(f"Merge {i+1}/{num_merges}: {best_pair} -> {new_id}")

        # Re-apply special tokens to vocabulary
        for token, idx in self.special_tokens.items():
            self.vocab[idx] = token.encode("utf-8")

    def _encode_chunk(self, chunk_bytes):
        """
        Fast rank-ordered pair merging on individual pre-tokenized chunks.
        """
        ids = list(chunk_bytes)
        while len(ids) >= 2:
            # Generate adjacent candidate pairs
            pairs = list(zip(ids, ids[1:]))
            
            # Find the candidate pair with the lowest merge token ID (trained earliest)
            pair = min(pairs, key=lambda p: self.merges.get(p, float("inf")))
            
            if pair not in self.merges:
                break
                
            idx = self.merges[pair]
            ids = merge(ids, pair, idx)
        return ids

    def encode(self, text, allowed_special="all"):
        """
        Encodes input string into token IDs with special token parsing.
        """
        if allowed_special == "all":
            active_specials = self.special_tokens
        elif allowed_special == "none":
            active_specials = {}
        elif isinstance(allowed_special, set):
            active_specials = {k: v for k, v in self.special_tokens.items() if k in allowed_special}
        else:
            raise ValueError(f"Invalid allowed_special configuration: {allowed_special}")

        if active_specials:
            # Build regex pattern to catch special tokens safely before BPE
            special_pattern = "(" + "|".join(re.escape(k) for k in active_specials.keys()) + ")"
            chunks = re.split(special_pattern, text)
        else:
            chunks = [text]

        ids = []
        for chunk in chunks:
            if chunk in active_specials:
                ids.append(active_specials[chunk])
            else:
                # Pre-tokenize sub-chunk and apply BPE merges
                sub_chunks = self.compiled_pattern.findall(chunk)
                for sub_chunk in sub_chunks:
                    chunk_bytes = sub_chunk.encode("utf-8")
                    ids.extend(self._encode_chunk(chunk_bytes))
        return ids

    def decode(self, ids):
        """
        Decodes token IDs back to a raw text string.
        """
        part_bytes = []
        for idx in ids:
            if idx in self.vocab:
                part_bytes.append(self.vocab[idx])
            elif idx in self.inverse_special_tokens:
                part_bytes.append(self.inverse_special_tokens[idx].encode("utf-8"))
            else:
                raise ValueError(f"Invalid Token ID: {idx}")

        return b"".join(part_bytes).decode("utf-8", errors="replace")

    def save(self, file_prefix):
        """
        Saves tokenizer model parameters and merge rules to disk.
        """
        model_file = file_prefix + ".model"
        with open(model_file, "w", encoding="utf-8") as f:
            f.write("gpt4-bpe v1\n")
            f.write(f"{self.pattern}\n")
            f.write(f"{len(self.special_tokens)}\n")
            for k, v in self.special_tokens.items():
                f.write(f"{k} {v}\n")
            for (p0, p1), idx in self.merges.items():
                f.write(f"{p0} {p1} {idx}\n")

    def load(self, model_file):
        """
        Loads pre-trained tokenizer state from a saved model file.
        """
        with open(model_file, "r", encoding="utf-8") as f:
            version = f.readline().strip()
            self.pattern = f.readline().strip()
            self.compiled_pattern = re.compile(self.pattern)

            num_special = int(f.readline().strip())
            self.special_tokens = {}
            self.inverse_special_tokens = {}
            for _ in range(num_special):
                line = f.readline().strip().split()
                k, v = line[0], int(line[1])
                self.special_tokens[k] = v
                self.inverse_special_tokens[v] = k

            self.merges = {}
            self.vocab = {i: bytes([i]) for i in range(256)}
            for k, v in self.special_tokens.items():
                self.vocab[v] = k.encode("utf-8")

            for line in f:
                if line.strip():
                    p0, p1, idx = map(int, line.strip().split())
                    self.merges[(p0, p1)] = idx
                    self.vocab[idx] = self.vocab[p0] + self.vocab[p1]


# -----------------------------------------------------------------------------
# GPT-4 Tokenizer Demonstration & Verification
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    code_corpus = """
    def compute_sum(a: int, b: int) -> int:
        # GPT-4 handles numbers (e.g. 123456789) by splitting every 1-3 digits!
        val_1 = 123
        val_2 = 456789
        return a + b + val_1 + val_2

    <|fim_prefix|>def add(a, b):<|fim_suffix|> return a + b<|fim_middle|>
    """

    # 1. Instantiate and Train
    tokenizer = GPT4BPETokenizer()
    tokenizer.train(code_corpus, vocab_size=320, verbose=False)

    # 2. Test Number Chunking & FIM Special Tokens
    sample_text = "<|fim_prefix|>val = 123456789<|fim_suffix|>"
    encoded_ids = tokenizer.encode(sample_text, allowed_special="all")
    decoded_text = tokenizer.decode(encoded_ids)

    print("Encoded Token IDs:")
    print(encoded_ids)
    print("\nDecoded Text:")
    print(decoded_text)

    # 3. Assert exact round-trip reconstruction
    assert sample_text == decoded_text, "Round-trip assertion failed!"
    print("\nRound-trip test passed successfully!")

Encoded Token IDs:
[100001, 118, 97, 108, 289, 32, 283, 285, 287, 100003]

Decoded Text:
<|fim_prefix|>val = 123456789<|fim_suffix|>

Round-trip test passed successfully!


---

In [27]:
import heapq
import json
import os
import regex as re

# Official GPT-4 / GPT-4o Regex Pre-tokenization Pattern
GPT4O_SPLIT_PATTERN = r"""(?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+"""

# Default Control / System Special Tokens
DEFAULT_SPECIAL_TOKENS = {
    "<|endoftext|>": 200000,
    "<|fim_prefix|>": 200001,
    "<|fim_middle|>": 200002,
    "<|fim_suffix|>": 200003,
    "<|endofprompt|>": 200004,
}


class Node:
    """Doubly Linked List Node for O(1) Token Insertion and Splice Operations."""
    __slots__ = ("val", "prev", "next")

    def __init__(self, val):
        self.val = val
        self.prev = None
        self.next = None


def get_stats(ids, counts=None):
    """Counts pair frequencies across integer sequence chunks."""
    counts = {} if counts is None else counts
    for p0, p1 in zip(ids, ids[1:]):
        pair = (p0, p1)
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge_list(ids, pair, idx):
    """Naive merge fallback used during trainer sequence updates."""
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


class OptimizedBPETokenizer:
    """
    Production-Ready Byte-Level BPE Tokenizer implementing:
    - GPT-4 / GPT-4o regex pre-tokenization boundary splitting
    - Fast Min-Heap + Doubly Linked List priority merge encoding
    - Comprehensive special token parsing
    - Complete save/load model serialization
    """

    def __init__(self, pattern=None):
        self.pattern = GPT4O_SPLIT_PATTERN if pattern is None else pattern
        self.compiled_pattern = re.compile(self.pattern)
        self.merges = {}  # tuple (p0, p1) -> merged_token_id
        self.vocab = {i: bytes([i]) for i in range(256)}  # int -> bytes
        self.special_tokens = {}
        self.inverse_special_tokens = {}

        self.register_special_tokens(DEFAULT_SPECIAL_TOKENS)

    def register_special_tokens(self, special_tokens_dict):
        """Registers system and control special tokens."""
        for token, idx in special_tokens_dict.items():
            self.special_tokens[token] = idx
            self.inverse_special_tokens[idx] = token
            self.vocab[idx] = token.encode("utf-8")

    def train(self, text, vocab_size=200000, verbose=False):
        """Trains BPE merge rules on raw input text up to target vocab_size."""
        assert vocab_size >= 256, "Vocabulary size must be at least 256."
        num_merges = vocab_size - 256

        # Step 1: Split raw text using regex boundaries
        text_chunks = self.compiled_pattern.findall(text)

        # Step 2: Convert text chunks into raw byte token ID lists
        ids_chunks = [list(chunk.encode("utf-8")) for chunk in text_chunks]

        # Step 3: Run iterative merge loop
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(256)}

        for i in range(num_merges):
            stats = {}
            for chunk in ids_chunks:
                get_stats(chunk, stats)

            if not stats:
                break

            best_pair = max(stats, key=stats.get)
            if stats[best_pair] < 1:
                break

            new_id = 256 + i
            ids_chunks = [merge_list(chunk, best_pair, new_id) for chunk in ids_chunks]

            self.merges[best_pair] = new_id
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]

            if verbose and (i + 1) % 1000 == 0:
                print(f"Learned Merge {i+1}/{num_merges}: {best_pair} -> {new_id}")

        for token, idx in self.special_tokens.items():
            self.vocab[idx] = token.encode("utf-8")

    def _encode_chunk_fast(self, chunk_bytes):
        """
        Fast priority-queue chunk encoding using Doubly Linked List + Min-Heap.
        Achieves optimal merge performance without scanning lists redundantly.
        """
        if len(chunk_bytes) < 2:
            return list(chunk_bytes)

        # 1. Build Doubly Linked List of raw bytes
        head = Node(chunk_bytes[0])
        curr = head
        for b in chunk_bytes[1:]:
            node = Node(b)
            curr.next = node
            node.prev = curr
            curr = node

        # 2. Populate Min-Heap with candidate merge pairs ordered by merge rank
        heap = []
        curr = head
        while curr and curr.next:
            pair = (curr.val, curr.next.val)
            if pair in self.merges:
                rank = self.merges[pair]
                # Heap elements: (rank, node_id, left_node)
                heapq.heappush(heap, (rank, id(curr), curr))
            curr = curr.next

        # 3. Process pairs in priority order (lowest merge rank first)
        while heap:
            rank, _, node = heapq.heappushpop(heap, (float("inf"), 0, None)) if not heap else heapq.heappop(heap)
            if node is None or node.next is None:
                continue

            # Verify that the candidate pair is still intact in the doubly-linked list
            pair = (node.val, node.next.val)
            if pair not in self.merges or self.merges[pair] != rank:
                continue

            # Merge nodes: Replace node value and splice out node.next
            node.val = rank
            merged_next = node.next.next
            node.next = merged_next
            if merged_next:
                merged_next.prev = node

            # Check and push newly created adjacent pairs around spliced node
            if node.prev:
                prev_pair = (node.prev.val, node.val)
                if prev_pair in self.merges:
                    heapq.heappush(heap, (self.merges[prev_pair], id(node.prev), node.prev))

            if node.next:
                next_pair = (node.val, node.next.val)
                if next_pair in self.merges:
                    heapq.heappush(heap, (self.merges[next_pair], id(node), node))

        # 4. Extract final token IDs from linked list
        ids = []
        curr = head
        while curr:
            ids.append(curr.val)
            curr = curr.next
        return ids

    def encode(self, text, allowed_special="all"):
        """Encodes string into integer token IDs handling special control tokens."""
        if allowed_special == "all":
            active_specials = self.special_tokens
        elif allowed_special == "none":
            active_specials = {}
        elif isinstance(allowed_special, set):
            active_specials = {k: v for k, v in self.special_tokens.items() if k in allowed_special}
        else:
            raise ValueError(f"Invalid allowed_special configuration: {allowed_special}")

        if active_specials:
            special_pattern = "(" + "|".join(re.escape(k) for k in active_specials.keys()) + ")"
            chunks = re.split(special_pattern, text)
        else:
            chunks = [text]

        ids = []
        for chunk in chunks:
            if chunk in active_specials:
                ids.append(active_specials[chunk])
            else:
                sub_chunks = self.compiled_pattern.findall(chunk)
                for sub_chunk in sub_chunks:
                    chunk_bytes = sub_chunk.encode("utf-8")
                    ids.extend(self._encode_chunk_fast(chunk_bytes))
        return ids

    def decode(self, ids):
        """Decodes integer token sequence back into a text string."""
        part_bytes = []
        for idx in ids:
            if idx in self.vocab:
                part_bytes.append(self.vocab[idx])
            elif idx in self.inverse_special_tokens:
                part_bytes.append(self.inverse_special_tokens[idx].encode("utf-8"))
            else:
                raise ValueError(f"Invalid token ID encountered: {idx}")

        return b"".join(part_bytes).decode("utf-8", errors="replace")

    def save(self, file_prefix):
        """Saves merge tables, patterns, and special tokens to disk."""
        model_file = file_prefix + ".model"
        with open(model_file, "w", encoding="utf-8") as f:
            f.write("gpt4o-bpe-engine v1\n")
            f.write(f"{self.pattern}\n")
            f.write(f"{len(self.special_tokens)}\n")
            for k, v in self.special_tokens.items():
                f.write(f"{k} {v}\n")
            for (p0, p1), idx in self.merges.items():
                f.write(f"{p0} {p1} {idx}\n")

    def load(self, model_file):
        """Loads trained merge tables and metadata from disk."""
        with open(model_file, "r", encoding="utf-8") as f:
            version = f.readline().strip()
            self.pattern = f.readline().strip()
            self.compiled_pattern = re.compile(self.pattern)

            num_special = int(f.readline().strip())
            self.special_tokens = {}
            self.inverse_special_tokens = {}
            for _ in range(num_special):
                line = f.readline().strip().split()
                k, v = line[0], int(line[1])
                self.special_tokens[k] = v
                self.inverse_special_tokens[v] = k

            self.merges = {}
            self.vocab = {i: bytes([i]) for i in range(256)}
            for k, v in self.special_tokens.items():
                self.vocab[v] = k.encode("utf-8")

            for line in f:
                if line.strip():
                    p0, p1, idx = map(int, line.strip().split())
                    self.merges[(p0, p1)] = idx
                    self.vocab[idx] = self.vocab[p0] + self.vocab[p1]


# -----------------------------------------------------------------------------
# End-to-End Verification Pipeline
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    sample_corpus = """
    def execute_pipeline(x: int) -> int:
        # Testing GPT-4o 3-digit number preservation: 123456789
        value = 123 + 456789
        return value * 42

    <|fim_prefix|>def run():<|fim_suffix|> return 0<|fim_middle|>
    """

    tokenizer = OptimizedBPETokenizer()
    tokenizer.train(sample_corpus, vocab_size=320, verbose=False)

    test_input = "def run(): <|fim_prefix|>val = 123456789<|fim_suffix|>"
    encoded = tokenizer.encode(test_input, allowed_special="all")
    decoded = tokenizer.decode(encoded)

    print("Encoded IDs:", encoded)
    print("Decoded Text:", decoded)

    assert test_input == decoded, "Round-trip assertion failed!"
    print("\nRound-trip invariant verified successfully.")

Encoded IDs: [268, 32, 114, 117, 110, 40, 41, 58, 32, 200001, 263, 108, 32, 61, 32, 274, 276, 278, 200003]
Decoded Text: def run(): <|fim_prefix|>val = 123456789<|fim_suffix|>

Round-trip invariant verified successfully.


Architectural Breakthroughs Beyond Standard BPE (2025–2026)

Recent advances eliminate traditional subword token boundaries:

- SuperBPE (Liu et al., 2025): Performs a second training pass that drops word-boundary restrictions altogether. This produces up to 33% fewer tokens per corpus and delivers a +4.0% average gain across downstream LLM benchmarks.

- BoundlessBPE (Schmidt et al., 2025): Allows multi-word phrases (e.g., "machine learning", "of the") to merge into single tokens in a single training pass, improving byte compression by up to 20%.

- Byte-Latent Transformers (BLT - Meta, 2025): Shifts away from fixed token vocabularies by learning dynamic, continuous latent representations directly on raw byte streams.